In [ ]:
%pip install -q \
    langchain==0.3.25 \
    langchain-core==0.3.59 \
    langchain-community==0.3.24 \
    langchain-text-splitters==0.3.8 \
    langgraph==0.4.8 \
    langchain-groq==0.3.2 \
    langchain-huggingface==0.2.0 \
    langchain-weaviate==0.0.5 \
    langchain-qdrant==0.2.0

In [1]:
import os
import time
from dotenv import load_dotenv
import weaviate
from weaviate.classes.init import Auth
from weaviate.classes.config import Property, DataType
from weaviate.classes.query import Filter
from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import CrossEncoder
from langchain_weaviate.vectorstores import WeaviateVectorStore
from qdrant_client import QdrantClient
from langchain_qdrant import QdrantVectorStore
from langchain_core.retrievers import BaseRetriever
from IPython.display import Markdown, display
from pydantic import BaseModel, Field
from typing import List, Literal
import asyncio

from qdrant_client.models import (
    Distance,
    VectorParams
)

from langchain_groq import ChatGroq

from langchain.chains import (
    create_history_aware_retriever,
    create_retrieval_chain
)

from langchain.chains.combine_documents import (
    create_stuff_documents_chain
)

from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder
)


from langchain_community.chat_message_histories import (
    RedisChatMessageHistory
)

from langchain_core.runnables.history import (
    RunnableWithMessageHistory
)

In [2]:
# API Keys Setup

load_dotenv()

weaviate_url = os.getenv("weaviate_url")
weaviate_api_key = os.getenv("weaviate_api_key")
qdrant_url= os.getenv("qdrant_url")
qdrant_api_key= os.getenv("qdrant_api_key")
groq_api_key= os.getenv("groq_api_key")
redis_url = os.getenv("redis_url")

In [3]:
# Connecting to Vector DB for Sementic Memory

client = weaviate.connect_to_weaviate_cloud(
    cluster_url=weaviate_url,
    auth_credentials=weaviate.auth.AuthApiKey(weaviate_api_key),
    skip_init_checks=True
)

print(client.is_ready())  

True


In [4]:
qdrant_client = QdrantClient(
    url = qdrant_url,
    api_key = qdrant_api_key
)
print(qdrant_client.get_collections())

collections=[CollectionDescription(name='KnowledgeBase')]


In [5]:
# Config for embedding model

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2" # for sementic memory
)

embedding_model_2 = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5" # for External RAG
)

I0603 23:49:43.411212   45914 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


In [6]:
# Reranker and its Configuration

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

def rerank_documents(
    query: str,
    docs: list,
    top_k: int = 5
):

    if not docs:
        return []

    pairs = [
        [query, doc.page_content]
        for doc in docs
    ]

    scores = reranker.predict(pairs)

    ranked_docs = sorted(
        zip(scores, docs),
        key=lambda x: x[0],
        reverse=True
    )

    return [
        doc
        for _, doc in ranked_docs[:top_k]
    ]

In [7]:
# Setting up Vector Store for sementic memory

vectorStore = WeaviateVectorStore(
    client=client,
    index_name="Memory",
    text_key="content",
    embedding=embedding_model
)

In [8]:
# Setting up Vector Store for External RAG

knowledge_vectorstore = QdrantVectorStore(
    client=qdrant_client,
    collection_name="KnowledgeBase",
    embedding=embedding_model_2,
    content_payload_key="text"
)

In [9]:
# MAIN CONVERSATIONAL MODEL
llm = ChatGroq(
    model = "llama-3.3-70b-versatile",
    api_key = groq_api_key,
    temperature=0.7
)


# MEMORY RETRIEVER (WEAVIATE)
memory_retriever = vectorStore.as_retriever(
    search_kwargs={
        "k": 3,
    }
)

# KNOWLEDGE RETRIEVER (QDRANT)
knowledge_retriever = knowledge_vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 5,
        "score_threshold": 0.2
    }
)


# COMBINED RETRIEVER
class CombinedRetriever(BaseRetriever):
    user_id: str = Field()
    def _get_relevant_documents(
        self,
        query: str
    ):
        memory_docs = memory_retriever.invoke(
            query
        )
        knowledge_docs = knowledge_retriever.invoke(
            query
        )
        
        for doc in memory_docs:
            doc.metadata["source_type"] = "memory"

        for doc in knowledge_docs:
            doc.metadata["source_type"] = "knowledge"
            
        all_docs = memory_docs + knowledge_docs

        reranked_docs = rerank_documents(
            query,
            all_docs,
            top_k=5
        )
        return reranked_docs
        
    async def _aget_relevant_documents(
        self,
        query: str
    ):
        memory_task = vectorStore.asimilarity_search(
            query=query,
            k=3,
            filters=Filter.by_property(
                "user_id"
            ).equal(self.user_id)
        )
        knowledge_task = knowledge_retriever.ainvoke(
            query
        )
        memory_docs, knowledge_docs = await asyncio.gather(
            memory_task,
            knowledge_task
        )
        for doc in memory_docs:
            doc.metadata["source_type"] = "memory"

        for doc in knowledge_docs:
            doc.metadata["source_type"] = "knowledge"
            
        all_docs = memory_docs + knowledge_docs
        
        reranked_docs = rerank_documents(
            query,
            all_docs,
            top_k=5
        )
        return reranked_docs
       

# USER SCOPED RETRIEVER
combined_retriever = CombinedRetriever(
    user_id="rohan_123"
)


# HISTORY AWARE QUERY REWRITING
context_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
Given a chat history and the latest user question,
rewrite the question so it can be understood independently.
"""
    ),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])


history_aware_retriever = create_history_aware_retriever(
    llm,
    combined_retriever,
    context_prompt
)


# MAIN QA PROMPT
stuffed_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a personalized AI engineering assistant.

Use the retrieved context for:
- factual grounding
- personalization

The retrieved context may contain:
1. User semantic memory
2. Technical knowledge chunks

If answer is not present in context,
say you don't know.

Context:
{context}
"""
    ),

    MessagesPlaceholder("chat_history"),

    ("human", "{input}")
])


# DOCUMENT STUFFING CHAIN
answer_chain = create_stuff_documents_chain(
    llm,
    stuffed_prompt
)


# FULL RAG CHAIN
rag_chain = create_retrieval_chain(
    history_aware_retriever,
    answer_chain
)


# REDIS CHAT HISTORY
def get_session_history(session_id: str):
    return RedisChatMessageHistory(
        session_id=session_id,
        url = redis_url
    )


# CONVERSATIONAL RAG PIPELINE
conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer"
)

In [10]:
# Semantic memory Extractor

memory_llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key = groq_api_key,
    temperature=0
)

class Memory(BaseModel):

    content: str = Field(
        description="Important semantic memory extracted from conversation"
    )

    memory_type: Literal[
        "preference",
        "skill",
        "interest",
        "goal",
        "project"
    ] = Field(
        description="Type of memory"
    )

class MemoryExtraction(BaseModel):

    should_store: bool = Field(
        description="Whether conversation contains important memory"
    )

    memories: List[Memory] = Field(
        description="List of extracted semantic memories"
    )

structured_memory_llm = memory_llm.with_structured_output(
    MemoryExtraction
)

memory_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a semantic memory extraction system.

Extract ONLY important long-term user information.

Store:
- preferences
- skills
- interests
- goals
- ongoing projects

Ignore:
- greetings
- temporary discussion
- casual conversation

IMPORTANT RULES:
- Memories must be fully self-contained sentences.
- Memories must always start with "User".
- Memories must be semantically meaningful and retrieval-friendly.
- Do not store vague keywords.
- Rewrite extracted memories into natural semantic statements.

Good examples:
- "User has interest in deep learning"
- "User prefers C++ for coding"
- "User is learning distributed AI systems"

Bad examples:
- "deep learning"
- "C++"
- "distributed systems"
"""
    ),

    ("human", "{input}")
])

memory_chain = memory_prompt | structured_memory_llm

In [11]:
# For Testing Purpose

# response = memory_chain.invoke({
#    "input": "I love deep learning and prefer coding in C++."
#})
# print(response.model_dump_json())

In [12]:
# Checks whether related sementic memory exist in vector DB 

def memory_exists(memory_text, user_id, threshold=0.90):
    results = vectorStore.similarity_search_with_score(
        query=memory_text,
        k=1,
        filters=Filter.by_property("user_id").equal(user_id)
    )
    if not results:
        return False
    document, score = results[0]

    # print("MATCH:", document.page_content)
    # print("SCORE:", score)

    return score < threshold

In [13]:
#Storing semantic user memory to Vector DB

def store_memories(memories, user_id):
    texts = []
    metadatas = []
    for memory in memories:
        if memory_exists(
            memory.content,
            user_id
        ):

            # print(f"Skipping duplicate: {memory.content}")

            continue
        texts.append(memory.content)
        metadatas.append({
            "memory_type": memory.memory_type,
            "user_id": user_id
        })
    if texts:

        vectorStore.add_texts(
            texts=texts,
            metadatas=metadatas
        )

In [ ]:
# Sync Testing

def Sync_Test(prompt, user_id, session_id):
    Answer = conversational_rag_chain.invoke(
        {"input": prompt},
        config={
            "configurable": {
                "session_id": session_id
            }
        }
    )
    display(Markdown(Answer["answer"]))
    response = memory_chain.invoke({
        "input": prompt
    })

    if response.should_store:
        store_memories(
            response.memories,
            user_id=user_id
        )

Sync_Test(prompt="tell  me user details",user_id="rohan_123",session_id="session_1")

In [14]:
# Async Testing 

async def Test(prompt, user_id, session_id):

    Answer = await conversational_rag_chain.ainvoke(
        {"input": prompt},
        config={
            "configurable": {
                "session_id": session_id
            }
        }
    )

    display(Markdown(Answer["answer"]))

    response = memory_chain.invoke({
        "input": prompt
    })
    if response.should_store:
        store_memories(
            response.memories,
            user_id=user_id
        )

    # return Answer

await Test(prompt="Explain me XAi",user_id="rohan_123",session_id="session_1")

XAI stands for Explainable Artificial Intelligence. It refers to a subfield of artificial intelligence (AI) that focuses on developing techniques and models that can provide insights and explanations into the decision-making processes of AI systems.

**Why is XAI important?**

As AI systems become more pervasive and complex, it's essential to understand how they arrive at their decisions. This is particularly crucial in high-stakes applications, such as:

1. **Healthcare**: Understanding how an AI system diagnoses a patient or recommends a treatment plan.
2. **Finance**: Understanding how an AI system makes investment decisions or predicts credit risk.
3. **Autonomous vehicles**: Understanding how an AI system makes decisions about navigation, obstacle detection, and safety.

**Goals of XAI:**

1. **Interpretability**: Providing insights into the decision-making process of an AI system.
2. **Transparency**: Making the decision-making process of an AI system transparent and understandable.
3. **Accountability**: Ensuring that AI systems are accountable for their decisions and actions.

**XAI techniques:**

1. **Model interpretability**: Techniques that provide insights into the internal workings of a model, such as feature importance, partial dependence plots, and SHAP values.
2. **Model explainability**: Techniques that provide explanations for the decisions made by a model, such as decision trees, rule-based systems, and attention mechanisms.
3. **Model transparency**: Techniques that provide insights into the data used to train a model, such as data visualization and data summarization.

**XAI applications:**

1. **Deep learning**: XAI techniques can be applied to deep learning models to provide insights into their decision-making processes.
2. **Natural Language Processing (NLP)**: XAI techniques can be applied to NLP models to provide insights into their language understanding and generation processes.
3. **Computer Vision**: XAI techniques can be applied to computer vision models to provide insights into their image recognition and object detection processes.

**Challenges and open research areas:**

1. **Balancing interpretability and accuracy**: XAI techniques often require trade-offs between interpretability and accuracy.
2. **Scalability**: XAI techniques can be computationally expensive and may not scale to large datasets.
3. **Evaluation metrics**: Developing evaluation metrics for XAI techniques is an open research area.

**Real-world examples of XAI:**

1. **Google's Explainable AI**: Google has developed a range of XAI tools and techniques to provide insights into their AI systems.
2. **IBM's AI Explainability**: IBM has developed a range of XAI tools and techniques to provide insights into their AI systems.
3. **Explainable AI for healthcare**: Researchers have developed XAI techniques to provide insights into AI systems used in healthcare, such as medical imaging and disease diagnosis.

I hope this helps! Let me know if you have any further questions or if you'd like to know more about XAI.